In [1]:
import matplotlib.pyplot as plt
import kagglehub
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:

# Download latest version
path = kagglehub.dataset_download("subhaditya/fer2013plus")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\margi\.cache\kagglehub\datasets\subhaditya\fer2013plus\versions\1


In [5]:
# Set up data transformations for training and testing
data_transforms = {
    "train": transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.RandomCrop(48, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ]),
    "test": transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
}

# Paths to dataset
train_dir = path + "/fer2013plus/fer2013/train"
test_dir = path + "/fer2013plus/fer2013/test"

# Load datasets using ImageFolder
train_dataset = datasets.ImageFolder(train_dir, transform=data_transforms["train"])
test_dataset = datasets.ImageFolder(test_dir, transform=data_transforms["test"])

print("Train and Test Datasets Loaded!")


Train and Test Datasets Loaded!


In [7]:
import numpy as np

# Get the frequency of each class
class_counts = np.bincount(train_dataset.targets)

# Calculate class weights inversely proportional to the frequency of each class
class_weights = 1. / class_counts
class_weights = torch.tensor(class_weights, dtype=torch.float32)

# Print class weights for verification
print("Class weights:", class_weights)


Class weights: tensor([4.0551e-04, 6.0606e-03, 5.2356e-03, 1.5337e-03, 1.3284e-04, 9.7012e-05,
        2.8458e-04, 2.8074e-04])


In [9]:
from torch.utils.data import WeightedRandomSampler

# Create sample weights for each data point in the training dataset
sample_weights = [class_weights[label] for label in train_dataset.targets]

# Create WeightedRandomSampler
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

# Create DataLoader using the sampler for stratified sampling
train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("DataLoader with Stratified Sampling created!")

DataLoader with Stratified Sampling created!


In [11]:
# Define the CNN architecture
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        
        # Fully connected layers
        self.fc1 = nn.Linear(in_features=128 * 6 * 6, out_features=256)
        self.fc2 = nn.Linear(in_features=256, out_features=128)
        self.fc3 = nn.Linear(in_features=128, out_features=8)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 6 * 6)  # Flatten the tensor
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Instantiate the model
model = CNN()
print("CNN Model Defined!")

CNN Model Defined!


In [13]:
# Define the loss function (CrossEntropyLoss) and optimizer (Adam)
criterion = nn.CrossEntropyLoss(weight=class_weights)  # Add class weights to loss function
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [14]:
# Train the model for a number of epochs
num_epochs = 3
for epoch in range(num_epochs):
    model.train()  # Set model to training mode
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader):
        optimizer.zero_grad()  # Zero the gradients

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Print loss every 100 batches
        if (i + 1) % 100 == 0:
            print(f"Epoch [{epoch + 1}/{num_epochs}], Step [{i + 1}/{len(train_loader)}], Loss: {running_loss / 100:.4f}")
            running_loss = 0.0

Epoch [1/3], Step [100/888], Loss: 1.4206
Epoch [1/3], Step [200/888], Loss: 1.3896
Epoch [1/3], Step [300/888], Loss: 1.3246
Epoch [1/3], Step [400/888], Loss: 1.2700
Epoch [1/3], Step [500/888], Loss: 1.1826
Epoch [1/3], Step [600/888], Loss: 1.1717
Epoch [1/3], Step [700/888], Loss: 1.1688
Epoch [1/3], Step [800/888], Loss: 1.1026
Epoch [2/3], Step [100/888], Loss: 1.0580
Epoch [2/3], Step [200/888], Loss: 0.9914
Epoch [2/3], Step [300/888], Loss: 0.9981
Epoch [2/3], Step [400/888], Loss: 0.9952
Epoch [2/3], Step [500/888], Loss: 0.9394
Epoch [2/3], Step [600/888], Loss: 0.8745
Epoch [2/3], Step [700/888], Loss: 0.8279
Epoch [2/3], Step [800/888], Loss: 0.8310
Epoch [3/3], Step [100/888], Loss: 0.8074
Epoch [3/3], Step [200/888], Loss: 0.7376
Epoch [3/3], Step [300/888], Loss: 0.6925
Epoch [3/3], Step [400/888], Loss: 0.6670
Epoch [3/3], Step [500/888], Loss: 0.6963
Epoch [3/3], Step [600/888], Loss: 0.6503
Epoch [3/3], Step [700/888], Loss: 0.5842
Epoch [3/3], Step [800/888], Loss:

In [16]:
from sklearn.metrics import f1_score

# Evaluate the model on the test set
model.eval()  # Set model to evaluation mode
correct = 0
total = 0
all_preds = []  # To store all predictions
all_labels = []  # To store all true labels

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # Store predictions and true labels
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate accuracy
accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")

# Calculate F1 score
f1 = f1_score(all_labels, all_preds, average='weighted')  # You can change average to 'macro' or 'micro' if needed
print(f"F1 Score (Weighted): {f1:.4f}")


Test Accuracy: 0.0609
F1 Score (Weighted): 0.0546


In [17]:
# Save the trained model to a file
torch.save(model.state_dict(), 'cnn_model.pth')
print("Model Saved as cnn_model.pth")

Model Saved as cnn_model.pth


In [18]:
# Prepare to count predictions for each class
classes = train_dataset.classes  # Get class names from the dataset
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

# Disable gradients for evaluation
with torch.no_grad():
    for data in test_loader:
        images, labels = data
        outputs = model(images)
        _, predictions = torch.max(outputs, 1)

        # Collect correct predictions for each class
        for label, prediction in zip(labels, predictions):
            label_name = classes[label]
            if label == prediction:
                correct_pred[label_name] += 1
            total_pred[label_name] += 1

# Print accuracy for each class
for classname, correct_count in correct_pred.items():
    if total_pred[classname] > 0:
        accuracy = 100 * float(correct_count) / total_pred[classname]
        print(f'Accuracy for class: {classname:10s} is {accuracy:.1f} %')
    else:
        print(f'No predictions for class: {classname}')

Accuracy for class: anger      is 10.4 %
Accuracy for class: contempt   is 82.4 %
Accuracy for class: disgust    is 73.7 %
Accuracy for class: fear       is 68.9 %
Accuracy for class: happiness  is 0.0 %
Accuracy for class: neutral    is 0.0 %
Accuracy for class: sadness    is 0.0 %
Accuracy for class: surprise   is 18.4 %
